In [ ]:
# ============================================================
# MEMBER C — CLIP FINE-TUNING + EMBEDDING GENERATION
# ============================================================

# =========================
# 1. INSTALL
# =========================
!pip install -q transformers accelerate open-clip-torch Pillow pandas tqdm scikit-learn

# =========================
# 2. IMPORTS
# =========================
import os
import gc
import random
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import open_clip

# =========================
# 3. PATHS
# =========================
METADATA_CSV  = "/kaggle/input/datasets/iharshsinha/vr-endterm-b-output/uploadB/metadata.csv"
CROP_ROOT     = "/kaggle/input/datasets/nipunv23/croppedimages/memberA_outputs/cropped_products"

OUTPUT_DIR    = "/kaggle/working/memberC_outputs"
CKPT_DIR      = os.path.join(OUTPUT_DIR, "checkpoints")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

PROGRESS_LOG  = os.path.join(OUTPUT_DIR, "progress_log.txt")
ERROR_LOG     = os.path.join(OUTPUT_DIR, "error_log.txt")

# =========================
# 4. CONSTANTS
# =========================
SEEDS            = [571, 129, 126, 591]

ALPHA_VALUES     = {
    "ablationA":    1.0,
    "ablationB_07": 0.7,
    "ablationB_05": 0.5,
    "ablationC_07": 0.7,
    "ablationC_05": 0.5,
}

CLIP_MODEL       = "ViT-B-32"
CLIP_PRETRAIN    = "openai"

BATCH_SIZE_EMB   = 128
BATCH_SIZE_TRAIN = 64
NUM_EPOCHS       = 5
LR               = 1e-5
TEMPERATURE      = 0.07
UNFREEZE_BLOCKS  = 4

# =========================
# 5. LOGGING
# =========================
prog_f  = open(PROGRESS_LOG, "a", buffering=1)
error_f = open(ERROR_LOG,    "a", buffering=1)

def log(msg):
    print(msg)
    prog_f.write(msg + "\n")
    prog_f.flush()

def log_error(msg):
    print(f"[ERROR] {msg}")
    error_f.write(msg + "\n")
    error_f.flush()

# =========================
# 6. GPU SETUP
# =========================
log("\n--- GPU Info ---")
log(f"CUDA available : {torch.cuda.is_available()}")
log(f"GPU count      : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    log(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 7. SEED UTILITY
# =========================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# =========================
# 8. LOAD DATA
# =========================
log("\nLoading metadata...")
meta_df = pd.read_csv(METADATA_CSV)
df      = meta_df.copy()

log(f"Columns in metadata    : {list(df.columns)}")
log(f"Total rows             : {len(df)}")
log(f"Caption nulls          : {df['caption'].isna().sum()}")

df["caption"] = df["caption"].fillna("")

def build_path(row):
    return os.path.join(CROP_ROOT, row["crop_class"], row["image_name"])

df["image_path"] = df.apply(build_path, axis=1)

exists_mask = df["image_path"].apply(os.path.exists)
log(f"Files found on disk    : {exists_mask.sum()} / {len(df)}")
df = df[exists_mask].reset_index(drop=True)
log(f"Rows after disk check  : {len(df)}")

# =========================
# 9. TRAIN / QUERY / GALLERY SPLIT
# =========================
log("\nBuilding train/query/gallery split...")

split_records = []

for crop_class in ["upper_body", "lower_body", "full_body"]:

    class_df     = df[df["crop_class"] == crop_class].copy()
    counts       = class_df.groupby("item_id").size()
    valid_items  = counts[counts >= 2].index.tolist()
    single_items = counts[counts == 1].index.tolist()

    log(f"  [{crop_class}] valid items: {len(valid_items)} | "
        f"single-image excluded: {len(single_items)}")

    set_seed(SEEDS[0])
    train_gallery_items, query_items = train_test_split(
        valid_items, test_size=0.15, random_state=SEEDS[0]
    )
    train_items, gallery_only_items = train_test_split(
        train_gallery_items, test_size=0.176,
        random_state=SEEDS[0]
    )

    for _, row in class_df.iterrows():
        if row["item_id"] in query_items:
            item_rows = class_df[class_df["item_id"] == row["item_id"]]
            query_row = item_rows.sample(1, random_state=SEEDS[0]).index[0]
            split = "query" if row.name == query_row else "gallery"
        elif row["item_id"] in train_items:
            split = "train"
        elif row["item_id"] in gallery_only_items:
            split = "gallery"
        else:
            split = "excluded"

        split_records.append({
            "image_name": row["image_name"],
            "item_id":    row["item_id"],
            "crop_class": crop_class,
            "split":      split
        })

split_df = pd.DataFrame(split_records)

SPLIT_CSV = os.path.join(OUTPUT_DIR, "split_info.csv")
split_df.to_csv(SPLIT_CSV, index=False)

log(f"\nSplit distribution:")
log(split_df.groupby(["crop_class", "split"]).size().to_string())

df = df.merge(split_df[["image_name", "split"]], on="image_name", how="left")

# =========================
# 10. DATASET CLASS
# =========================
class FashionCropDataset(Dataset):

    def __init__(self, dataframe, preprocess, split=None):
        if split:
            self.df = dataframe[
                dataframe["split"] == split
            ].reset_index(drop=True)
        else:
            self.df = dataframe.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            img = self.preprocess(img)
        except Exception:
            img = torch.zeros(3, 224, 224)
        return {
            "image":      img,
            "caption":    str(row["caption"]),
            "item_id":    str(row["item_id"]),
            "crop_class": str(row["crop_class"]),
            "image_name": str(row["image_name"]),
        }

# =========================
# 11. CONTRASTIVE LOSS
# =========================
class ContrastiveLoss(nn.Module):

    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, item_ids, crop_classes):
        embeddings = F.normalize(embeddings, dim=-1)
        n          = embeddings.shape[0]

        sim = torch.matmul(embeddings, embeddings.T) / self.temperature

        pos_mask = torch.zeros(n, n, dtype=torch.bool, device=embeddings.device)
        for i in range(n):
            for j in range(n):
                if (i != j
                        and item_ids[i] == item_ids[j]
                        and crop_classes[i] == crop_classes[j]):
                    pos_mask[i, j] = True

        if pos_mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True).to(embeddings.device)

        diag_mask = ~torch.eye(n, dtype=torch.bool, device=embeddings.device)

        loss  = 0.0
        count = 0

        for i in range(n):
            pos_indices = pos_mask[i].nonzero(as_tuple=True)[0]
            if len(pos_indices) == 0:
                continue
            denom = torch.logsumexp(sim[i][diag_mask[i]], dim=0)
            for j in pos_indices:
                loss  += -(sim[i, j] - denom)
                count += 1

        if count == 0:
            return torch.tensor(0.0, requires_grad=True).to(embeddings.device)

        return loss / count

# =========================
# 12. FREEZE / UNFREEZE
# =========================
def freeze_clip_vision(model):
    for param in model.visual.parameters():
        param.requires_grad = False

def unfreeze_last_n_blocks(model, n=4):
    freeze_clip_vision(model)
    blocks = model.visual.transformer.resblocks
    for block in blocks[-n:]:
        for param in block.parameters():
            param.requires_grad = True
    if hasattr(model.visual, "ln_post"):
        for param in model.visual.ln_post.parameters():
            param.requires_grad = True
    if hasattr(model.visual, "proj") and model.visual.proj is not None:
        model.visual.proj.requires_grad = True

def freeze_clip_text(model):
    for param in model.transformer.parameters():
        param.requires_grad = False
    for param in model.token_embedding.parameters():
        param.requires_grad = False
    if hasattr(model, "ln_final"):
        for param in model.ln_final.parameters():
            param.requires_grad = False

# =========================
# 13. FINE-TUNING FUNCTION
# =========================
def fine_tune_clip(df, seed):

    log(f"\n{'='*60}")
    log(f"Fine-tuning CLIP | seed={seed}")
    log(f"{'='*60}")

    set_seed(seed)

    model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL, pretrained=CLIP_PRETRAIN
    )
    model = model.to(DEVICE)

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        log(f"Using {torch.cuda.device_count()} GPUs")

    base_model = model.module if hasattr(model, "module") else model

    unfreeze_last_n_blocks(base_model, UNFREEZE_BLOCKS)
    freeze_clip_text(base_model)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    log(f"Trainable params: {trainable:,} / {total:,}")

    train_dataset = FashionCropDataset(df, preprocess, split="train")
    train_loader  = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE_TRAIN,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True,
    )
    log(f"Train batches: {len(train_loader)} | Train samples: {len(train_dataset)}")

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,
        weight_decay=0.01
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS
    )
    criterion = ContrastiveLoss(temperature=TEMPERATURE)

    best_loss = float("inf")
    best_ckpt = os.path.join(CKPT_DIR, f"best_clip_seed{seed}.pt")

    for epoch in range(NUM_EPOCHS):

        model.train()
        epoch_loss  = 0.0
        batch_count = 0

        for batch in tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{NUM_EPOCHS} seed={seed}"
        ):
            images       = batch["image"].to(DEVICE)
            item_ids     = batch["item_id"]
            crop_classes = batch["crop_class"]

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                img_features = base_model.encode_image(images)
                img_features = F.normalize(img_features.float(), dim=-1)
                loss         = criterion(img_features, item_ids, crop_classes)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss  += loss.item()
            batch_count += 1

        scheduler.step()

        avg_loss = epoch_loss / max(batch_count, 1)
        log(f"  Epoch {epoch+1} | loss={avg_loss:.4f}")

        epoch_ckpt = os.path.join(CKPT_DIR, f"clip_seed{seed}_epoch{epoch+1}.pt")
        torch.save(base_model.state_dict(), epoch_ckpt)

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(base_model.state_dict(), best_ckpt)
            log(f"  --> New best saved: {best_ckpt}")

    log(f"Fine-tuning done. Best loss: {best_loss:.4f}")
    return best_ckpt, preprocess

# =========================
# 14. EMBEDDING GENERATION
# =========================
def generate_embeddings(df, model, preprocess, alpha, label):

    log(f"\nGenerating embeddings: {label} | alpha={alpha}")

    dataset = FashionCropDataset(df, preprocess, split=None)
    loader  = DataLoader(
        dataset,
        batch_size=BATCH_SIZE_EMB,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )

    base_model = model.module if hasattr(model, "module") else model
    base_model.eval()

    all_embeddings  = []
    all_image_names = []
    errors          = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Embedding {label}"):
            images   = batch["image"].to(DEVICE)
            captions = batch["caption"]

            try:
                with torch.amp.autocast("cuda"):
                    img_emb = base_model.encode_image(images)
                    img_emb = F.normalize(img_emb.float(), dim=-1)

                    if alpha < 1.0:
                        tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
                        tokens    = tokenizer(captions).to(DEVICE)
                        txt_emb   = base_model.encode_text(tokens)
                        txt_emb   = F.normalize(txt_emb.float(), dim=-1)
                        fused     = alpha * img_emb + (1 - alpha) * txt_emb
                    else:
                        fused = img_emb

                    fused = F.normalize(fused, dim=-1)

                all_embeddings.append(fused.cpu().numpy())
                all_image_names.extend(batch["image_name"])

            except Exception as e:
                log_error(f"Embedding batch error [{label}]: {e}")
                errors += 1
                continue

    if not all_embeddings:
        log_error(f"No embeddings generated for {label}!")
        return None, []

    embeddings_np = np.concatenate(all_embeddings, axis=0)
    log(f"  Shape: {embeddings_np.shape} | Errors: {errors}")

    out_path = os.path.join(OUTPUT_DIR, f"embeddings_{label}.npy")
    np.save(out_path, embeddings_np)

    with open(out_path, "rb") as f:
        os.fsync(f.fileno())

    log(f"  Saved: {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)")

    return embeddings_np, all_image_names

# =========================
# 15. MAIN PIPELINE
# =========================

# ---- STEP 1: ABLATION A + B (frozen CLIP) ----
log("\n" + "="*60)
log("STEP 1: FROZEN CLIP EMBEDDINGS (Ablation A + B)")
log("="*60)

seed_embeddings = {k: [] for k in ["ablationA", "ablationB_07", "ablationB_05"]}
image_names_ref = None

for seed in SEEDS:
    set_seed(seed)
    log(f"\n--- Frozen CLIP | seed={seed} ---")

    model_frozen, _, preprocess_frozen = open_clip.create_model_and_transforms(
        CLIP_MODEL, pretrained=CLIP_PRETRAIN
    )
    model_frozen = model_frozen.to(DEVICE)

    freeze_clip_vision(model_frozen)
    freeze_clip_text(model_frozen)

    if torch.cuda.device_count() > 1:
        model_frozen = nn.DataParallel(model_frozen)

    for ablation_key in ["ablationA", "ablationB_07", "ablationB_05"]:
        alpha = ALPHA_VALUES[ablation_key]
        emb, img_names = generate_embeddings(
            df, model_frozen, preprocess_frozen,
            alpha, label=f"{ablation_key}_seed{seed}"
        )
        if emb is not None:
            seed_embeddings[ablation_key].append(emb)
            if image_names_ref is None:
                image_names_ref = img_names

    del model_frozen
    torch.cuda.empty_cache()
    gc.collect()

# average across seeds
log("\nAveraging frozen embeddings across seeds...")
for ablation_key in ["ablationA", "ablationB_07", "ablationB_05"]:
    stack    = np.stack(seed_embeddings[ablation_key], axis=0)
    mean_emb = stack.mean(axis=0)
    std_emb  = stack.std(axis=0)

    norms    = np.linalg.norm(mean_emb, axis=1, keepdims=True)
    mean_emb = mean_emb / np.clip(norms, 1e-8, None)

    out_path = os.path.join(OUTPUT_DIR, f"embeddings_{ablation_key}.npy")
    np.save(out_path, mean_emb)

    std_path = os.path.join(OUTPUT_DIR, f"embeddings_{ablation_key}_std.npy")
    np.save(std_path, std_emb)

    log(f"  {ablation_key}: shape={mean_emb.shape} | mean±std saved")

# ---- STEP 2: FINE-TUNE CLIP (Ablation C) ----
log("\n" + "="*60)
log("STEP 2: FINE-TUNED CLIP EMBEDDINGS (Ablation C)")
log("="*60)

seed_embeddings_C = {"ablationC_07": [], "ablationC_05": []}

for seed in SEEDS:
    set_seed(seed)

    best_ckpt_path, preprocess_ft = fine_tune_clip(df, seed)

    model_ft, _, _ = open_clip.create_model_and_transforms(
        CLIP_MODEL, pretrained=CLIP_PRETRAIN
    )
    model_ft.load_state_dict(
        torch.load(best_ckpt_path, map_location=DEVICE)
    )
    model_ft = model_ft.to(DEVICE)

    if torch.cuda.device_count() > 1:
        model_ft = nn.DataParallel(model_ft)

    for ablation_key in ["ablationC_07", "ablationC_05"]:
        alpha = ALPHA_VALUES[ablation_key]
        emb, _ = generate_embeddings(
            df, model_ft, preprocess_ft,
            alpha, label=f"{ablation_key}_seed{seed}"
        )
        if emb is not None:
            seed_embeddings_C[ablation_key].append(emb)

    del model_ft
    torch.cuda.empty_cache()
    gc.collect()

# average across seeds
log("\nAveraging fine-tuned embeddings across seeds...")
for ablation_key in ["ablationC_07", "ablationC_05"]:
    stack    = np.stack(seed_embeddings_C[ablation_key], axis=0)
    mean_emb = stack.mean(axis=0)
    std_emb  = stack.std(axis=0)

    norms    = np.linalg.norm(mean_emb, axis=1, keepdims=True)
    mean_emb = mean_emb / np.clip(norms, 1e-8, None)

    out_path = os.path.join(OUTPUT_DIR, f"embeddings_{ablation_key}.npy")
    np.save(out_path, mean_emb)

    std_path = os.path.join(OUTPUT_DIR, f"embeddings_{ablation_key}_std.npy")
    np.save(std_path, std_emb)

    log(f"  {ablation_key}: shape={mean_emb.shape} saved")

# copy best model for Member D
best_for_D       = os.path.join(CKPT_DIR, f"best_clip_seed{SEEDS[0]}.pt")
final_model_path = os.path.join(OUTPUT_DIR, "best_clip_model.pt")
shutil.copy(best_for_D, final_model_path)
log(f"\nBest model for Member D saved: {final_model_path}")

# ---- STEP 3: METADATA WITH EMBEDDINGS ----
log("\n" + "="*60)
log("STEP 3: SAVING METADATA WITH EMBEDDINGS")
log("="*60)

meta_out                    = df.copy()
meta_out["embedding_index"] = range(len(meta_out))

META_OUT_CSV = os.path.join(OUTPUT_DIR, "metadata_with_embeddings.csv")
meta_out.to_csv(META_OUT_CSV, index=False)

with open(META_OUT_CSV, "rb") as f:
    os.fsync(f.fileno())

log(f"Saved: {META_OUT_CSV}")

# =========================
# 16. FINAL VERIFICATION
# =========================
log("\n" + "="*60)
log("FINAL VERIFICATION")
log("="*60)

log(f"\nFiles in {OUTPUT_DIR}:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        log(f"  {fname}  ({os.path.getsize(fpath)/1e6:.1f} MB)")

log(f"\nCheckpoints in {CKPT_DIR}:")
for fname in sorted(os.listdir(CKPT_DIR)):
    fpath = os.path.join(CKPT_DIR, fname)
    log(f"  {fname}  ({os.path.getsize(fpath)/1e6:.1f} MB)")

log("\nEmbedding shapes:")
for key in ALPHA_VALUES.keys():
    path = os.path.join(OUTPUT_DIR, f"embeddings_{key}.npy")
    if os.path.exists(path):
        arr = np.load(path)
        log(f"  {key}: {arr.shape}")

log("\nMetadata preview:")
final_meta = pd.read_csv(META_OUT_CSV)
log(f"  Rows: {len(final_meta)}")
log(f"  Splits:\n{final_meta.groupby(['crop_class','split']).size().to_string()}")

log("\nMember C pipeline complete.")
prog_f.close()
error_f.close()